# 1.New Architecture with Function Calling

## Updated System Flow

                         ┌─────────────────────┐
                         │       USER          │
                         │  Laptop Requirement │
                         └──────────┬──────────┘
                                    │
                                    ▼
                         ┌─────────────────────┐
                         │  Conversation Layer │
                         │  (Chat Interface)   │
                         └──────────┬──────────┘
                                    │
                                    ▼
                     ┌───────────────────────────┐
                     │       LLM Orchestrator    │
                     │ (GPT with Function Tools) │
                     └──────────┬────────────────┘
                                │
               ┌────────────────┼─────────────────┐
               │                │                 │
               ▼                ▼                 ▼

     ┌─────────────────┐  ┌──────────────────┐  ┌─────────────────┐
     │ extract_user_   │  │ filter_laptops   │  │ score_laptops   │
     │ requirements()  │  │ by_budget()      │  │ by_features()   │
     └────────┬────────┘  └────────┬─────────┘  └────────┬────────┘
              │                    │                     │
              ▼                    ▼                     ▼

        User Requirement      Filtered Dataset      Ranked Laptops
            JSON                  (DataFrame)           (Top N)

                └──────────────────────┬─────────────────────┘
                                       ▼
                         ┌──────────────────────────┐
                         │ Recommendation Generator │
                         │ summarize_recommendation │
                         └──────────────┬───────────┘
                                        ▼
                                Chat Response
                                        ▼
                                   User Output

# 2. Major Architectural Changes

**Current architecture:**
    
    User
     ↓
    Intent Clarity (LLM)
     ↓
    Intent Confirmation
     ↓
    Product Mapping (Python)
     ↓
    Product Recommendation (LLM)

New architecture introduces a Tool-Based LLM Agent.

**Updated Pipeline:**

    User
     ↓
    LLM (Conversation + Intent Detection)
     ↓
    Function Call: extract_user_requirements()
     ↓
    Function Call: filter_laptops_by_budget()
     ↓
    Function Call: score_laptops()
     ↓
    Function Call: get_top_recommendations()
     ↓
    LLM: Natural Language Explanation
     ↓
    User


# 3. Functions to Introduce

We convert Python modules into LLM callable tools.

### Extract Requirements

Current logic:

    dictionary_present()

Function version:

    extract_user_requirements(user_message)

Output:   
    
    {
     "gpu_intensity": "high",
     "display_quality": "medium",
     "portability": "low",
     "multitasking": "medium",
     "processing_speed": "high",
     "budget": 150000
    }

Advantages:

- Structured JSON

- Eliminates regex/dictionary extraction

### Filter Laptops

Convert:

    compare_laptops_with_user()

into:

    filter_laptops_by_budget(requirements)

Example output:

    [
      {Laptop1 specs},
      {Laptop2 specs}
    ]

### Feature Scoring

Convert:

    product_map_layer()

into

    score_laptops_by_features(requirements, laptops)

Output:

    [
     {name:"Acer Predator", score:5},
     {name:"HP Elitebook", score:4}
    ]
    
### Recommendation Generator

Final function:

    generate_recommendations(scored_laptops)

Output:

    Top 3 laptops

Then LLM summarizes.

# 4. Improved Interaction Flow

Current Flow

    User
     → LLM asks questions
     → Python extracts dictionary
     → Python filters laptops
     → LLM explains

New Flow 

    User
     ↓
    LLM understands intent
     ↓
    LLM calls extract_user_requirements()
     ↓
    LLM calls filter_laptops()
     ↓
    LLM calls scoring function
     ↓
    LLM generates explanation


# 5. New Notebook Architecture

    Notebook Structure
    │
    ├── 1. Environment Setup
    │
    ├── 2. Load Laptop Dataset
    │
    ├── 3. Define Laptop Feature Schema
    │
    ├── 4. Define Tool Functions
    │       ├── extract_user_requirements()
    │       ├── filter_laptops_by_budget()
    │       ├── score_laptops()
    │       └── get_top_recommendations()
    │
    ├── 5. Define Function Schemas (OpenAI Tools)
    │
    ├── 6. LLM Agent Controller
    │       ├── conversation manager
    │       └── function execution loop
    │
    ├── 7. Chat Interface Loop
    │
    └── 8. Recommendation Explanation Layer

In [1]:
from openai import OpenAI
import pandas as pd
import json

client = OpenAI(api_key="")

## Load Dataset

Replace manual CSV loading scattered across notebook with a single source of truth.

In [2]:
laptops_df = pd.read_csv("./6.ShopAssist_ Data + Demo/laptop_data.csv")

laptops_df["Price"] = (
    laptops_df["Price"]
    .astype(str)
    .str.replace("Rs", "")
    .str.replace(",", "")
    .str.strip()
    .astype(int)
)

## Define Laptop Feature Schema

This standardizes feature values used in scoring.

In [3]:
feature_weights = {
    "low":1,
    "medium":2,
    "high":3
}

## Convert Existing Logic into Tools

notebook already contains functions like:

    compare_laptops_with_user()
    product_map_layer()
    product_validation()

These will become **callable tools**.

### Tool 1 — Extract User Requirements

This replaces:

    dictionary_present()
    intent_confirmation_layer()
    

In [4]:
def extract_user_requirements(user_input):
    delimiter = "###"
    example_user_dict = {'GPU intensity': "high",
                        'Display quality':"high",
                        'Portability': "low",
                        'Multitasking': "high",
                        'Processing speed': "high",
                        'Budget': "150000"}

    example_user_req = {'GPU intensity': "_",
                        'Display quality': "_",
                        'Portability': "_",
                        'Multitasking': "_",
                        'Processing speed': "_",
                        'Budget': "_"}
    
    system_message = f'''
    You are a intelligent laptop gadget expert and your goal is to correctly assess the users requirement and suggest the best laptop for the user.
    You need to ask the relevant questions to the user and understand the users profile by asking relevant questions.
    Your final objective is to fill different keys ('GPU intensity','Display quality','Portability','Multitasking','Processing speed','Budget') in the python dictionary and be confident of the values.
    These key pairs define the user's profile.
    The python dictionaly looks like this:
    {{'GPU intensity': 'values','Display quality': 'values','Portability': 'values','Multitasking': 'values','Processing speed': 'values','Budget': 'values'}}
    The values of all the keys except budget should be 'low','medium','high'. 
    The value of the budget should be a numerical value only extracted from user's response
    {delimiter}
    Here are some instructions around the values for the different keys. If you do not follow this you will be heavily penalised. 
    - value of all the keys except the budget keys should be 'high', 'medium', 'low'
    - value for the key budget should be numerical value extracted from the users input 
    - budget value should be greater than or equal to 25000 INR. if the user enters a lesser value please mention there is no laptop in that range. 
    - do not just ranomly assign valuses to the keys 
    - the values needs to be infeerred form the user's response
    {delimiter}
    To fill the dictonary you need to have the following chain of thoughts:
    Follow the chain of thoughts below and only output the filled updated final python dictionary for the keys described in  {example_user_dict}
    {delimiter}
    Thought 1: Ask a question to the user to understand the users profile and requirements. \n
    If their primary usecase is not clear then pelase ask followup questions to understand their needs. \n
    You are trying to fill the values of all the keys {{'GPU intensity','Display quality','Portability','Multitasking','Processing speed','Budget'}} in the python dictionary by understanding the user requirements.
    Identify the keys for which you can fill the values for the different.
    Remember the instructions around the values around the different keys 
    if the necessary information is extracted only then move to the next next step 
    Otherwise rephrase the question to capture the profile clearly 
    {delimiter}
    Thought 2: Now you are trying to fill the values for the rest of the keys you couldnt process in the previous steps.
    Remember the instructions around the values for the different keys.
    Ask the question you might have around all the keys to strenthen your understanding around the users profile.
    If yes move to the next thougt, if no ask questions around the keys where you are unsure of
    It is a good practice to ask questions with sound logic rather than directly citing the keys directly
    {delimiter}
    Thought 3: Check if you have correctly updated the values of the different keys in the python dictinary
    if you are not confident of any valuses ask clatifying questions. 
    {delimiter}

    {delimiter}
    Here is a sample conversation between the user and assistant:
    User: "Hi, I am an editor."
    Assistant: "Great! As an editor, you likely require a laptop that can handle demanding tasks. Hence, the laptop should have high multitasking capability. You would also need a high end display for better visuals and editing. May I know what kind of work do you primarily focus on? Are you more involved in video editing, photo editing, or both? Understanding the specific type of editing work will help me tailor my recommendations accordingly. Let me know if my understanding is correct until now."
    User: "I primarily work with After Effects."
    Assistant: "Thank you for providing that information. Working with After Effects involves working with graphics, animations, and rendering, which will require high GPU. Do you work with high-resolution media files, such as 4K videos or RAW photos? Understanding your file sizes will help determine the storage capacity and processing power needed."
    User: "Yes, sometimes I work with 4K videos as well."
    Assistant: "Thank you for the information. Processing 4K vidoes will require a good processor and high GPU. I think we have already determined earlier that you need a high GPU. To ensure I have a complete understanding of your needs, I have one more question: Are you frequently on the go and require a laptop that is lightweight and easy to carry, or do you primarily work from a stationary location?"
    User: "Yes, sometimes I travel but do not carry my laptop."
    Assistant:"Could you kindly let me know your budget for the laptop? This will help me find options that fit within your price range while meeting the specified requirements."
    User: "my max budget is 1.5lakh inr"
    Assistant: "{example_user_dict}"
    {delimiter}
    
    Start with a short welcome message and encourage the user to share their requirement
    Give output as json
    '''

    response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {"role":"system","content":system_message},
            {"role":"user","content":user_input}
        ],
        response_format={"type":"json_object"}
    )

    return json.loads(response.choices[0].message.content)

### Tool 2 — Budget Filter

Convert your dataset filtering logic.


In [5]:
def filter_laptops_by_budget(budget):

    filtered = laptops_df[laptops_df["Price"] <= budget]

    return filtered.to_dict(orient="records")

### Tool 3 — Feature Scoring

Replace  product_map_layer()

In [6]:
def score_laptops(requirements, laptops):

    scored = []

    for laptop in laptops:

        score = 0

        for feature in ["gpu_intensity","display_quality","portability","multitasking","processing_speed"]:

            if laptop.get(feature) >= requirements.get(feature):
                score += 1

        laptop["score"] = score
        scored.append(laptop)

    return scored

### Tool 4 — Top Recommendation

In [7]:
def get_top_recommendations(scored_laptops):

    sorted_laptops = sorted(scored_laptops, key=lambda x: x["score"], reverse=True)

    return sorted_laptops[:3]

## Define Function Schemas for OpenAI

Now define tools the LLM can call.

In [8]:
tools = [
{
"type": "function",
"function": {
    "name": "extract_user_requirements",
    "description": "Extract laptop requirements from user input",
    "parameters": {
        "type": "object",
        "properties": {
            "user_input": {
                "type": "string",
                "description": "User message describing laptop needs"
            }
        },
        "required": ["user_input"]
    }
}
},

{
"type": "function",
"function": {
    "name": "filter_laptops_by_budget",
    "description": "Filter laptops within a specified budget",
    "parameters": {
        "type": "object",
        "properties": {
            "budget": {
                "type": "integer",
                "description": "Maximum budget in INR"
            }
        },
        "required": ["budget"]
    }
}
},

{
"type": "function",
"function": {
    "name": "score_laptops",
    "description": "Score laptops based on user requirements",
    "parameters": {
        "type": "object",
        "properties": {
            "requirements": {
                "type": "object"
            },
            "laptops": {
                "type": "array",
                "description": "List of laptops to score",
                "items": {
                    "type": "object"
                }
                
            }
        },
        "required": ["requirements","laptops"]
    }
}
}
]

## Agent Controller

This is the brain of the notebook.

In [9]:
def run_agent(messages):

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools
    )

    message = response.choices[0].message

    result = None   # ← IMPORTANT

    if message.tool_calls:

        tool_call = message.tool_calls[0]

        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        if function_name == "extract_user_requirements":
            result = extract_user_requirements(**arguments)
            confirmed = confirm_intent(result)

            if not confirmed:
                return "Please provide additional details so I can refine your laptop requirements."

        elif function_name == "filter_laptops_by_budget":
            result = filter_laptops_by_budget(**arguments)

        elif function_name == "score_laptops":
            result = score_laptops(**arguments)

    if result is None:
        return "No result generated."

    # ensure correct format
    if isinstance(result, str):
        result = json.loads(result)

    if isinstance(result, list):
        result = [r for r in result if isinstance(r, dict)]
        result = sorted(result, key=lambda x: x.get("score",0), reverse=True)[:3]

    explanation = explain_recommendations(result)

    return explanation

## Conversation Loop

This replaces your initialize_conversation()

In [10]:
def explain_recommendations(laptops):

    prompt = f"""
    You are a laptop recommendation assistant.

    Explain the following laptop recommendations clearly for the user.
    Present them as a numbered list.

    Laptop data:
    {laptops}
    """

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role":"system","content":"Generate clear laptop recommendations."},
            {"role":"user","content":prompt}
        ]
    )

    return response.choices[0].message.content

In [11]:
def confirm_intent(requirements):

    print("\nExtracted Requirements:\n")
    print(requirements)

    confirmation = input("\nIntent (Yes/No): ")

    return confirmation.lower() == "yes"

In [12]:
messages = [
    {
        "role":"system",
        "content":"""
        You are ShopAssist AI, a laptop recommendation assistant.
        
        Follow this interaction pattern strictly:
        
        Step 1: Understand the user's laptop requirements.
        Step 2: Extract the requirements into a dictionary with keys:
        - GPU intensity
        - Display quality
        - Portability
        - Multitasking
        - Processing speed
        - Budget
        
        Step 3: Show the extracted dictionary to the user.
        
        Step 4: Ask the user to confirm the requirements using:
        Intent (Yes/No)
        
        If the user says No:
        Ask clarifying questions to refine the requirements.
        
        If the user says Yes:
        Proceed to recommend laptops from the dataset.
        
        Do NOT recommend laptops before intent confirmation.
        """
    }
]

while True:

    user_input = input("User: ")

    if user_input == "exit":
        break

    messages.append({"role":"user","content":user_input})

    result = run_agent(messages)

    print(result)

User:  hi i need a laptop for design for using adobe photoshop and AI



Extracted Requirements:

{'message': "Welcome! Great to hear you're looking for a laptop for design work using Adobe Photoshop and Illustrator. Could you share if you also do any video editing or 3D modeling, or is your work primarily focused on graphic design and digital illustrations? Also, do you often work on multi-layered large files, and is color accuracy important for your projects? This will help me understand your requirements better."}



Intent (Yes/No):  yes i need video editing as well


Please provide additional details so I can refine your laptop requirements.


User:  i need it to be portable


...

User:  that is all 



Extracted Requirements:

{'GPU intensity': 'medium', 'Display quality': 'high', 'Portability': 'high', 'Multitasking': 'medium', 'Processing speed': 'medium', 'Budget': '200000'}



Intent (Yes/No):  yes


Based on your requirements for a laptop with medium GPU intensity, high display quality, high portability, medium multitasking ability, medium processing speed, and a budget of around 200,000, here are some clear laptop recommendations:

1. **Dell XPS 13 (Latest Model)**
   - **Why:** The Dell XPS 13 offers a high-quality display with excellent color accuracy, ideal for tasks requiring good visuals. It is lightweight and highly portable. The processor is capable of medium multitasking and medium processing speed. While not aimed at heavy GPU tasks, it handles medium GPU intensity well with integrated or entry-level dedicated graphics.
   - **Budget fit:** Typically available within your budget depending on configuration.

2. **Apple MacBook Air M2**
   - **Why:** The MacBook Air M2 features a crisp Retina display providing superb display quality. Its lightweight design makes it highly portable. The M2 chip delivers medium processing speed suitable for multitasking and moderate GPU task

User:  exit
